## Competition for nutrients in ENVIRONMENT (only compounds in metabolomics!!): consumption overlap

In [1]:
import pandas as pd
import json
import seaborn as sns
import numpy as np

import networkx as nx

from pyvis.network import Network

In [2]:
with open("../data/Venturelli2018/generated/metid2metname.json") as json_file:
    metid2metname = json.load(json_file)

In [3]:
results_part1 = pd.read_csv("../output/Venturelli2018/results_communities_12_1000s_part1.tsv",sep="\t")
results_part2 = pd.read_csv("../output/Venturelli2018/results_communities_12_1000s_part2.tsv",sep="\t")
results = pd.concat([results_part1,results_part2]).reset_index(drop=True)

results["mass_rate*frequency"] = results["frequency"]*results["mass_rate"]

In [4]:
communities_12_exp_abundance = pd.read_csv("../data/Venturelli2018/generated/communities_12_exp_abundance.tsv",sep="\t",index_col=[0,1])
communities_12_exp_abundance

6
ER-  Blautia_hydrogenotrophica_DSM_10507     0.0109
     Collinsella_aerofaciens_ATCC_25986      0.0255
     Bacteroides_uniformis_ATCC_8492         0.3809
     Prevotella_copri_CB7_DSM_18205          0.0003
     Bacteroides_ovatus_ATCC_8483            0.1343
...                                             ...
NONE Eggerthella_lenta_DSM_2243              0.0168
     Faecalibacterium_prausnitzii_A2_165     0.2393
     Clostridium_hiranonis_TO_931_DSM_13275  0.0212
     Desulfovibrio_piger_ATCC_29098          0.0019
     Eubacterium_rectale_ATCC_33656          0.0086

[144 rows x 1 columns]

In [5]:
def centrality_metrics(network_data,cutoff=0,SP_centrality=True):
    """
    SP_centrality: centrality meassures based on shortest path
    """
    # Betweenness and closeness
    G = nx.Graph()
    G.add_nodes_from(list(set(list(network_data.index.get_level_values(0).unique())+list(network_data.index.get_level_values(1).unique()))))
    
    if SP_centrality:
        edges = [(index[0],index[1],{"weight":1/data} ) for index,data in network_data.items() if data>cutoff]
    
    else:
        edges = [(index[0],index[1],{"weight":data} ) for index,data in network_data.items() if data>cutoff]
        
    G.add_edges_from(edges)
    
    if SP_centrality:
        return pd.DataFrame({"betweeness (1/data)":nx.centrality.betweenness_centrality(G,weight="weight"),
                     "closeness (1/data)":nx.centrality.closeness_centrality(G,distance="weight")})
    else:
        return pd.DataFrame({"degree (data)":dict(G.degree(weight="weight")),
                 "eigenvector (data)":nx.centrality.eigenvector_centrality(G,weight="weight")})


In [6]:
## From https://stackoverflow.com/questions/25668828/how-to-create-colour-gradient-in-python
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

def colorFader(c1,c2,mix=0): #fade (linear interpolate) from color c1 (at mix=0) to c2 (mix=1)
    c1=np.array(mpl.colors.to_rgb(c1))
    c2=np.array(mpl.colors.to_rgb(c2))
    return mpl.colors.to_hex((1-mix)*c1 + mix*c2)


In [7]:
b1='#f8cdcf' #bright pink
b2='#fa4f56' #red

c1='#ffffff' #white
c2='#0b5394' #blue

In [8]:
def spearman_corr_network(network_data,closeness_centrality_met,cutoff=0):
    net = Network()

    nodes = list(set(list(network_data.index.get_level_values(0).unique())+list(network_data.index.get_level_values(1).unique())))
    color = [colorFader(b1,b2,closeness_centrality_met[node]) for node in nodes]
    value = [(closeness_centrality_met[node]*10)**2 for node in nodes]
    net.add_nodes(nodes,color=color,value=value)
    
    
    maximum_corr = network_data.max()
    for index,value in network_data.items():
        if value>cutoff:
            net.add_edge(index[0],index[1],width=(value**2)*10,color=colorFader(c1,c2,value/maximum_corr)) 

    net.barnes_hut(gravity=-1000, central_gravity=0.1, spring_length=150, spring_strength=0.0015, damping=0.09, overlap=0.8)


    template = net.templateEnv.get_template(net.path)
    net.template = template
    return net

In [9]:
def count_overlap_network(network_data,closeness_centrality_met,cutoff=0):
    net = Network()
    
    maximum_centrality = pd.Series(closeness_centrality_met).max()

    nodes = list(set(list(network_data.index.get_level_values(0).unique())+list(network_data.index.get_level_values(1).unique())))
    
    
    color = [colorFader(b1,b2,closeness_centrality_met[node]/maximum_centrality) for node in nodes]
    value = [(closeness_centrality_met[node]*10)**2 for node in nodes]
    net.add_nodes(nodes,color=color,value=value)

    maximum_overlap = network_data.max()
    for index,value in network_data.items():
        if value>cutoff:
            net.add_edge(index[0],index[1],width=(value**2)/10,color=colorFader(c1,c2,value/maximum_overlap))

    net.barnes_hut(gravity=-1000, central_gravity=0.1, spring_length=150, spring_strength=0.0015, damping=0.09, overlap=0.8)


    template = net.templateEnv.get_template(net.path)
    net.template = template
    return net

### Spearman correlation

In [10]:
results_env = results[(results.donor=="environment")]
results_env = results_env[["receiver","compound","frequency"]]
# Find mean of frequencies
results_grouped_rec_env = results_env[(results_env.compound.isin(metid2metname.keys()))].groupby(["receiver","compound"]).mean()

**Correlation matrix**

In [11]:
network_data = results_grouped_rec_env.unstack().T.corr(method="spearman")
network_data = network_data.mask(np.tril(np.ones(network_data.shape)).astype(np.bool)).stack() 

**Network topological properties**


- **Betweeness_centrality**: Sum of shortest paths that pass through node i
- **Degree centrality**: The fraction of nodes that node i is connected to. 
- **Eigenvector centrality**: Higher if node i is connected to other well connected notes.

In [12]:
centrality_metrics_SP = centrality_metrics(network_data).sort_values("closeness (1/data)")
centrality_metrics_SP

,betweeness (1/data),closeness (1/data)
Desulfovibrio_piger_ATCC_29098,0.000000,0.548312
Bacteroides_vulgatus_ATCC_8482,0.000000,0.651607
Bacteroides_thetaiotaomicron_VPI_5482,0.000000,0.658894
Eggerthella_lenta_DSM_2243,0.000000,0.679201
Eubacterium_rectale_ATCC_33656,0.000000,0.701139
Prevotella_copri_CB7_DSM_18205,0.000000,0.718214
Bacteroides_ovatus_ATCC_8483,0.000000,0.730442
Clostridium_hiranonis_TO_931_DSM_13275,0.000000,0.738129
Faecalibacterium_prausnitzii_A2_165,0.000000,0.738891
Blautia_hydrogenotrophica_DSM_10507,0.018182,0.744236


In [13]:
centrality_metrics(network_data,SP_centrality=False).sort_values("degree (data)")

,degree (data),eigenvector (data)
Desulfovibrio_piger_ATCC_29098,5.592535,0.205809
Bacteroides_thetaiotaomicron_VPI_5482,7.287479,0.268973
Bacteroides_vulgatus_ATCC_8482,7.412181,0.265463
Eggerthella_lenta_DSM_2243,7.658254,0.281192
Eubacterium_rectale_ATCC_33656,8.139289,0.293842
Prevotella_copri_CB7_DSM_18205,8.286197,0.295316
Clostridium_hiranonis_TO_931_DSM_13275,8.294449,0.296141
Bacteroides_ovatus_ATCC_8483,8.450403,0.300692
Faecalibacterium_prausnitzii_A2_165,8.481889,0.303614
Blautia_hydrogenotrophica_DSM_10507,8.593634,0.303175


In [14]:
closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"].to_dict()

In [15]:
net = spearman_corr_network(network_data,closeness_centrality_met=closeness_centrality_met)
net.show("example.html")

example.html


**Network topological properties (correlation>0.9!!!)**


- **Betweeness_centrality**: Sum of shortest paths that pass through node i
- **Degree centrality**: The fraction of nodes that node i is connected to. 
- **Eigenvector centrality**: Higher if node i is connected to other well connected notes.

In [16]:
cutoff = 0.9

In [17]:
centrality_metrics_SP = centrality_metrics(network_data,cutoff=cutoff).sort_values("closeness (1/data)")
centrality_metrics_SP

,betweeness (1/data),closeness (1/data)
Clostridium_hiranonis_TO_931_DSM_13275,0.000000,0.000000
Desulfovibrio_piger_ATCC_29098,0.000000,0.193680
Faecalibacterium_prausnitzii_A2_165,0.000000,0.197813
Blautia_hydrogenotrophica_DSM_10507,0.163636,0.245507
Collinsella_aerofaciens_ATCC_25986,0.163636,0.251433
Eubacterium_rectale_ATCC_33656,0.000000,0.285134
Prevotella_copri_CB7_DSM_18205,0.254545,0.291908
Bacteroides_vulgatus_ATCC_8482,0.000000,0.296759
Bacteroides_ovatus_ATCC_8483,0.254545,0.297366
Bacteroides_uniformis_ATCC_8492,0.436364,0.346888


In [18]:
centrality_metrics(network_data,SP_centrality=False,cutoff=0.9).sort_values("degree (data)")

,degree (data),eigenvector (data)
Clostridium_hiranonis_TO_931_DSM_13275,0.000000,3.640410e-48
Desulfovibrio_piger_ATCC_29098,0.908295,1.522106e-01
Faecalibacterium_prausnitzii_A2_165,0.918297,2.990775e-02
Eubacterium_rectale_ATCC_33656,1.841886,8.974716e-02
Bacteroides_thetaiotaomicron_VPI_5482,1.939914,2.304725e-01
Prevotella_copri_CB7_DSM_18205,2.000000,9.725543e-02
Bacteroides_uniformis_ATCC_8492,2.754790,4.445479e-01
Bacteroides_vulgatus_ATCC_8482,2.757256,4.977498e-01
Bacteroides_ovatus_ATCC_8483,2.760670,4.982257e-01
Blautia_hydrogenotrophica_DSM_10507,2.768241,4.223554e-01


**Network (cutoff correlation = 0.9 !!)**

In [19]:
closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"].to_dict()

In [20]:
net = spearman_corr_network(network_data,closeness_centrality_met=closeness_centrality_met,cutoff=cutoff)
net.show("example.html")

example.html


### Overlap count (compound consumed with high frequency)

In [21]:
results_env = results[(results.donor=="environment")]
results_env = results_env[["receiver","compound","frequency"]]
# Find mean of frequencies
results_grouped_rec_env = results_env[(results_env.compound.isin(metid2metname.keys()))].groupby(["receiver","compound"]).mean()

**Count (cutoff frequency=0.5!!)**

In [22]:
count_same = results_grouped_rec_env["frequency"]

count_same = count_same.unstack().fillna(0)>0.5
count_same = count_same.astype(int)
count_same = count_same.dot(count_same.T)

network_data = count_same.mask(np.tril(np.ones(count_same.shape)).astype(np.bool)).stack() 

maximum = network_data.max()

**Network topological properties**


- **Betweeness_centrality**: Sum of shortest paths that pass through node i
- **Degree centrality**: The fraction of nodes that node i is connected to. 
- **Eigenvector centrality**: Higher if node i is connected to other well connected notes.

In [23]:
centrality_metrics_SP = centrality_metrics(network_data).sort_values("closeness (1/data)")
centrality_metrics_SP

,betweeness (1/data),closeness (1/data)
Prevotella_copri_CB7_DSM_18205,0.000000,4.025506
Eggerthella_lenta_DSM_2243,0.000000,4.624625
Bacteroides_uniformis_ATCC_8492,0.000000,5.239085
Blautia_hydrogenotrophica_DSM_10507,0.000000,6.074951
Collinsella_aerofaciens_ATCC_25986,0.000000,6.169597
Bacteroides_thetaiotaomicron_VPI_5482,0.000000,6.384155
Bacteroides_vulgatus_ATCC_8482,0.000000,6.553191
Desulfovibrio_piger_ATCC_29098,0.000000,6.755882
Bacteroides_ovatus_ATCC_8483,0.018182,7.100410
Faecalibacterium_prausnitzii_A2_165,0.000000,7.199660


In [24]:
centrality_metrics(network_data,SP_centrality=False).sort_values("degree (data)")

,degree (data),eigenvector (data)
Prevotella_copri_CB7_DSM_18205,44.0,0.167027
Eggerthella_lenta_DSM_2243,54.0,0.207369
Bacteroides_uniformis_ATCC_8492,61.0,0.231295
Blautia_hydrogenotrophica_DSM_10507,75.0,0.291098
Bacteroides_vulgatus_ATCC_8482,75.0,0.280476
Bacteroides_thetaiotaomicron_VPI_5482,75.0,0.279200
Collinsella_aerofaciens_ATCC_25986,75.0,0.289210
Bacteroides_ovatus_ATCC_8483,83.0,0.307907
Desulfovibrio_piger_ATCC_29098,83.0,0.317246
Clostridium_hiranonis_TO_931_DSM_13275,89.0,0.336778


**Network**

In [25]:
closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"].to_dict()

net = count_overlap_network(network_data,closeness_centrality_met,cutoff=0)
net.show("example.html")

example.html


**Cutoff=7!!**

In [26]:
centrality_metrics_SP = centrality_metrics(network_data,cutoff=7).sort_values("closeness (1/data)")
centrality_metrics_SP

,betweeness (1/data),closeness (1/data)
Prevotella_copri_CB7_DSM_18205,0.000000,0.000000
Eggerthella_lenta_DSM_2243,0.000000,0.000000
Bacteroides_uniformis_ATCC_8492,0.000000,3.356740
Bacteroides_vulgatus_ATCC_8482,0.000000,4.420287
Collinsella_aerofaciens_ATCC_25986,0.000000,4.586348
Bacteroides_thetaiotaomicron_VPI_5482,0.045455,4.652944
Blautia_hydrogenotrophica_DSM_10507,0.000000,4.667093
Desulfovibrio_piger_ATCC_29098,0.000000,4.971867
Bacteroides_ovatus_ATCC_8483,0.100000,5.231432
Clostridium_hiranonis_TO_931_DSM_13275,0.100000,5.785714


In [27]:
centrality_metrics(network_data,SP_centrality=False,cutoff=7).sort_values("degree (data)")

,degree (data),eigenvector (data)
Prevotella_copri_CB7_DSM_18205,0.0,3.949354e-27
Eggerthella_lenta_DSM_2243,0.0,3.949354e-27
Bacteroides_uniformis_ATCC_8492,17.0,5.154260e-02
Bacteroides_vulgatus_ATCC_8482,26.0,1.249892e-01
Bacteroides_thetaiotaomicron_VPI_5482,35.0,1.261554e-01
Collinsella_aerofaciens_ATCC_25986,44.0,3.369218e-01
Bacteroides_ovatus_ATCC_8483,45.0,1.975936e-01
Blautia_hydrogenotrophica_DSM_10507,46.0,3.500086e-01
Desulfovibrio_piger_ATCC_29098,50.0,3.750645e-01
Clostridium_hiranonis_TO_931_DSM_13275,59.0,4.010407e-01


In [35]:
closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"].to_dict()

net = count_overlap_network(network_data,closeness_centrality_met,cutoff=5)
net.show("example.html")

example.html


In [32]:
closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"].to_dict()

net = count_overlap_network(network_data,closeness_centrality_met,cutoff=6)
net.show("example.html")

example.html


Main one

In [29]:
closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"].to_dict()

net = count_overlap_network(network_data,closeness_centrality_met,cutoff=7)
net.show("example.html")

example.html


In [31]:
closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"].to_dict()

net = count_overlap_network(network_data,closeness_centrality_met,cutoff=8)
net.show("example.html")

example.html
